# combining data

In [2]:
import geopandas as gpd
import pandas as pd

In [ ]:
import geopandas as gpd
import pandas as pd

#gdf1 = gpd.read_file("data/AMPHIBIANS/AMPHIBIANS_PART1.shp")
#gdf2 = gpd.read_file("data/AMPHIBIANS/AMPHIBIANS_PART2.shp")

gdf1= gpd.read_file("data/MAMMALS/MAMMALS_PART1.shp")
gdf2 = gpd.read_file("data/MAMMALS/MAMMALS_PART2.shp")

gdf = pd.concat([gdf1, gdf2], ignore_index=True)



#gdf = pd.concat([gdf, gdf3], ignore_index=True)
#gdf = pd.concat([gdf1, gdf4], ignore_index=True)


##### save file ####
#gdf.to_file("output.gpkg", driver="GPKG")


# Getting common names

In [ ]:
import pandas as pd
import sqlite3



df = gpd.read_file("output.gpkg")

df = df[(df['marine'] == "false")]


# -----------------------------
# Connect to ITIS SQLite DB
# -----------------------------
DB_PATH = "data/itisSqlite032626/ITIS.sqlite"
conn = sqlite3.connect(DB_PATH)


# -----------------------------
# Cache for speed
# -----------------------------
tsn_cache = {}
common_cache = {}


# -----------------------------
# Get TSN from scientific name
# -----------------------------
def get_tsn(scientific_name: str):
    if scientific_name in tsn_cache:
        return tsn_cache[scientific_name]

    query = """
    SELECT tsn
    FROM taxonomic_units
    WHERE complete_name = ?
    LIMIT 1
    """

    cur = conn.execute(query, (scientific_name,))
    row = cur.fetchone()

    tsn = row[0] if row else None
    tsn_cache[scientific_name] = tsn

    return tsn


# -----------------------------
# Get common names from TSN
# -----------------------------
def get_common_names(tsn: int):
    if tsn in common_cache:
        return common_cache[tsn]

    if tsn is None:
        return []

    query = """
    SELECT vernacular_name
    FROM vernaculars
    WHERE tsn = ?
    """

    cur = conn.execute(query, (tsn,))
    rows = cur.fetchall()

    names = [r[0] for r in rows if r[0]]

    common_cache[tsn] = names
    return names


# -----------------------------
# Full lookup function
# -----------------------------
def lookup_common_names(scientific_name: str):
    tsn = get_tsn(scientific_name)
    return get_common_names(tsn)


# -----------------------------
# Apply to DataFrame
# -----------------------------
def add_common_names_column(df: pd.DataFrame, col="sci_name"):
    df = df.copy()
    df["common_names"] = df[col].apply(lookup_common_names)
    return df





df = add_common_names_column(df)


#create display names

def get_display_name(row):
    names = row["common_names"]
    
    # If it's a real list and not empty
    if isinstance(names, list) and len(names) > 0:
        return names[0]
    
    # If it's a string that looks like a list
    if isinstance(names, str) and names.startswith("["):
        try:
            import ast
            parsed = ast.literal_eval(names)
            if isinstance(parsed, list) and len(parsed) > 0:
                return parsed[0]
        except:
            pass

    # Fallback
    return row["sci_name"]

gdf["display_name"] = gdf.apply(get_display_name, axis=1)

gdf.to_file("output_common_names_fixed.gpkg", driver="GPKG")

In [ ]:
import numpy as np

gdf = gpd.read_file("output_common_names.gpkg")

gdf["display_name"] = gdf["common_names"]
gdf.loc[gdf["common_names"].isna() | (gdf["common_names"] == ""), "display_name"] = gdf["sci_name"]

gdf.to_file("output_common_names_fixed.gpkg", driver="GPKG")



In [ ]:
############## Combining species ##########################

gdf = gpd.read_file("output_common_names_fixed.gpkg")

gdf_combined = gdf.dissolve(by="sci_name").reset_index()

gdf_combined.to_file("output_common_names_fixed_combined.gpkg", driver="GPKG")

In [3]:
import geopandas as gpd

gdf = gpd.read_file("output_common_names_fixed.gpkg")

gdf_combined = gdf.dissolve(
    by="sci_name",
    aggfunc={
        "SHAPE_Leng": "sum",
        "SHAPE_Area": "sum",
        "tax_comm": lambda x: "; ".join(x.dropna().unique()),
        "dist_comm": lambda x: "; ".join(x.dropna().unique()),
        "presence": "first",
        "origin": "first",
        "seasonal": "first",
        "compiler": "first",
        "yrcompiled": "first",
        "citation": "first",
        "family": "first",
        "genus": "first",
        "category": "first",
        "marine": "max",
        "terrestria": "max",
        "freshwater": "max",
        "common_names": "first",
        "display_name": "first",
    }
).reset_index()


gdf_combined.to_file("output_common_names_fixed_combined.gpkg", driver="GPKG")

In [ ]:
##### Reduce size of geometry ###

gdf = gpd.read_file("output_common_names_fixed_combined.gpkg")

gdf["geometry"] = gdf["geometry"].simplify(tolerance=0.01)


gdf.to_file("output_common_names_fixed_combined_reduced.gpkg", driver="GPKG")

In [53]:
import geopandas as gpd
import pandas as pd
import urllib.parse



gdf = gpd.read_file("output_common_names_fixed_combined_reduced.gpkg")

gdf = gdf[(gdf['marine'] == "false")]


gdf = gdf.reset_index(drop=True)
gdf["uid"] = gdf.index.astype(str)




#add google link for easy searching of sci_name
gdf["google_link"] = gdf["sci_name"].apply(
    lambda x: f'<a href="https://www.google.com/search?q={urllib.parse.quote(x)}" target="_blank">Search on Google</a>'
)


gdf["wiki_link"] = gdf["sci_name"].apply(
    lambda x: f'<a href="https://en.wikipedia.org/wiki/Special:Search/{urllib.parse.quote(x)}" target="_blank">Wikipedia</a>'
)


gdf["iucn_link"] = gdf["sci_name"].apply(
    lambda x: f'<a href="https://www.iucnredlist.org/search?query={urllib.parse.quote(x)}" target="_blank">IUCN Red List</a>'
)


## Calculate average area of habitat polygons to order them
gdf["n_parts"] = gdf.geometry.apply(
    lambda geom: len(geom.geoms) if geom.geom_type == "MultiPolygon" else 1
)

gdf["avg_part_area"] = gdf["SHAPE_Area"] / gdf["n_parts"]




ex = gdf[(gdf['marine'] == "false") & (gdf['category'].isin(["EX", "EW"]))]
cr = gdf[(gdf['marine'] == "false") & (gdf['category'] == "CR")]
en = gdf[(gdf['marine'] == "false") & (gdf['category'] == "EN")]
vu = gdf[(gdf['marine'] == "false") & (gdf['category'] == "VU")]


cr = cr.sort_values("SHAPE_Area", ascending=False)
en = en.sort_values("SHAPE_Area", ascending=False)
ex = ex.sort_values("SHAPE_Area", ascending=False)
vu = vu.sort_values("SHAPE_Area", ascending=False)


cr.to_file("cr_combined.gpkg")
en.to_file("en_combined.gpkg")
ex.to_file("ex_combined.gpkg")
vu.to_file("vu_combined.gpkg")


#gdf["centroid"] = gdf.to_crs(epsg=3857).centroid.to_crs(gdf.crs)


#points_cr = cr.set_geometry("centroid")
#points_en = en.set_geometry("centroid")
#points_ex = ex.set_geometry("centroid")









In [ ]:
import geopandas as gpd
cr = gpd.read_file("cr.gpkg")
en = gpd.read_file("en.gpkg")
ex = gpd.read_file("ex.gpkg")


In [ ]:
import folium

#endangered.explore()

#points_tmp = tmp.set_geometry("centroid")

map_tooltip = ["sci_name", "category", "yrcompiled"]
map_popup = ["common_names", "sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"]
draggable = True,
map_style_kwds = {
    "color": "black",   # border (consistent)
    "weight": 0.2,
    "fillOpacity": 0.2,
}

m = tmp.explore(
    color="red",
    tooltip=["sci_name", "yrcompiled"],
    popup=["sci_name", "yrcompiled"],
    legend_name="test",
    style_kwds={
        "color": "black",     # border color
        "weight": 1,          # border thickness
        "fillOpacity": 0.5,    # fill transparency
        #"fill": False
    }
)



#points_tmp.explore(
#    m=m,
#    name="EN points (Endangered)",
#    tooltip=map_tooltip,
#    popup=map_popup,
#    marker_type="circle_marker"
#)

#m = endangered.explore()







m.save("tmp.html")

# plotting

In [ ]:
#endangered.explore()

m = endangered.explore(
    color="red",
    tooltip=["sci_name", "yrcompiled"],
    popup=["sci_name", "yrcompiled"],
    legend = True,
    style_kwds={
        "color": "black",     # border color
        "weight": 0.1,          # border thickness
        "fillOpacity": 0.1,    # fill transparency
        #"fill": False
    }
)

#m = endangered.explore()
m.save("endangered.html")

# Multiple layers map

In [ ]:
import folium


map_tooltip = ["common_names", "sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"]
map_popup = ["common_names", "sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"]
map_style_kwds = {
    "color": "black",   # border (consistent)
    "weight": 0.2,
    "fillOpacity": 0.2,
}




m = ex.explore(
    name="EX (Extinct)",
    tooltip=map_tooltip,
    popup=map_popup,
    style_kwds={**map_style_kwds, "fillColor": "black"}
)

folium.map.CustomPane("polygons", z_index=400).add_to(m)
folium.map.CustomPane("points", z_index=650).add_to(m)

cr.explore(
    m=m,
    name="CR (Critically endangered)",
    tooltip=map_tooltip,
    popup=map_popup,
    style_kwds={**map_style_kwds, "fillColor": "rgba(238, 62, 50, 1)"}
)

en.explore(
    m=m,
    name="EN (Endangered)",
    show=False,
    tooltip=map_tooltip,
    popup=map_popup,
    style_kwds={**map_style_kwds, "fillColor": "purple" }
)

points_cr.explore(
    m=m,
    name="CR points (Critically endangered)",
    tooltip=map_tooltip,
    popup=map_popup,
    marker_type="circle_marker"
)

points_en.explore(
    m=m,
    name="EN points (Endangered)",
    tooltip=map_tooltip,
    popup=map_popup,
    marker_type="circle_marker"
)

###### crashes the map :( 
#vu.explore(
#    m=m,
#    name="VU (vulnerable)",
#    show=False,
#    tooltip=map_tooltip,
#    popup=map_popup,
#    style_kwds={**map_style_kwds, "fillColor": "rgba(27, 138, 90, 1)"}
#)



folium.LayerControl().add_to(m)

m.save("stacked_by_area_layered_endangered.html")

# Folium

In [54]:
import geopandas as gpd
cr = gpd.read_file("cr.gpkg")
en = gpd.read_file("en.gpkg")
ex = gpd.read_file("ex.gpkg")


In [55]:
import geopandas as gpd
import webbrowser
import folium

#cr = cr[:10]
#en = en[:10]


m = folium.Map([0, 0], zoom_start=2, control_scale=True, tiles="cartodbpositron")


ex_group = folium.FeatureGroup(name="Extinct", show=False).add_to(m)
cr_group = folium.FeatureGroup(name="Critically endangered", show=True).add_to(m)
en_group = folium.FeatureGroup(name="Endangered", show=False).add_to(m)

#map_tooltip = ["sci_name", "category", "yrcompiled"]
#map_popup = ["sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"]

############### ex
folium.GeoJson(
    data=ex,
    smooth_factor=2,

    style_function=lambda feature: {
        "fillColor": "#404028",
        "color": "black",
        "weight": 0.1,
        "fillOpacity": 0.3,
    },

    popup=folium.GeoJsonPopup(
        fields=["common_names", "sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"],
        aliases=["Common names:", "Scientific name:", "Endangerment catagory:", "Year compiled:", "Google:", "Wikipeida:", "IUCN:"],
        localize=True
    ),

    tooltip=folium.GeoJsonTooltip(
        fields=["common_names"],
        aliases=["Common names:"],
        localize=True,
        sticky=True
    ),

    highlight_function=lambda feature: {
        "fillColor": "#ffff00",
        "color": "black",
        "weight": 3,
        "fillOpacity": 0.7,
    },
    popup_keep_highlighted=True,
).add_to(ex_group)



######### cr
folium.GeoJson(
    cr,
    smooth_factor=2,

    style_function=lambda feature: {
        "fillColor": "#FF0000",
        "color": "black",
        "weight": 0.1,
        "fillOpacity": 0.3,
    },
    popup=folium.GeoJsonPopup(
        fields=["common_names", "sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"],
        aliases=["Common names:", "Scientific name:", "Endangerment catagory:", "Year compiled:", "Google:", "Wikipeida:", "IUCN:"],
        localize=True
    ),

    tooltip=folium.GeoJsonTooltip(
        fields=["common_names"],
        aliases=["Common names:"],
        localize=True,
        sticky=True
    ),

    highlight_function=lambda feature: {
        "fillColor": "#ffff00",
        "color": "black",
        "weight": 3,
        "fillOpacity": 0.7,
    },
    popup_keep_highlighted=True,
).add_to(cr_group)


############## en
folium.GeoJson(
    data= en,
    smooth_factor=2,
    
    style_function=lambda feature: {
        "fillColor": "#0033FF",
        "color": "black",
        "weight": 0.1,
        "fillOpacity": 0.1,
    },

    popup=folium.GeoJsonPopup(
        fields=["common_names", "sci_name", "category", "yrcompiled", "google_link", "wiki_link", "iucn_link"],
        aliases=["Common names:", "Scientific name:", "Endangerment catagory:", "Year compiled:", "Google:", "Wikipeida:", "IUCN:"],
        localize=True
    ),

    tooltip=folium.GeoJsonTooltip(
        fields=["common_names"],
        aliases=["Common names:"],
        localize=True,
        sticky=True
    ),

    highlight_function=lambda feature: {
        "fillColor": "#ffff00",
        "color": "black",
        "weight": 3,
        "fillOpacity": 0.7,
    },
    popup_keep_highlighted=True,
).add_to(en_group)





folium.LayerControl().add_to(m)

m.save("tmpmap.html")
webbrowser.open("tmpmap.html")


True

# Better Folium

In [6]:
import geopandas as gpd
cr = gpd.read_file("cr_combined.gpkg")
en = gpd.read_file("en_combined.gpkg")
ex = gpd.read_file("ex_combined.gpkg")
#vu = gpd.read_file("vu_combined.gpkg")


import ast

cr["common_names"] = cr["common_names"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

cr["common_names"] = cr["common_names"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)


#cr["geometry"] = cr["geometry"].simplify(tolerance=0.01)
#en["geometry"] = en["geometry"].simplify(tolerance=0.01)
#ex["geometry"] = ex["geometry"].simplify(tolerance=0.01)
#vu["geometry"] = vu["geometry"].simplify(tolerance=10)


In [7]:
import geopandas as gpd
import webbrowser
import folium
from branca.element import Template, MacroElement

#cr = cr[:10]
#en = en[:10]



m = folium.Map([0, 0], zoom_start=2, control_scale=True, tiles="cartodbpositron")

def make_layer(data, group, fill_color="#FFFF00"):
    return folium.GeoJson(
        data=data,
        smooth_factor=2,

        style_function=lambda feature: {
            "fillColor": fill_color,
            "color": "black",
            "weight": 0.2,
            "fillOpacity": 0.1,
        },

        popup=folium.GeoJsonPopup(
            fields=[
                str("common_names"),  
                "sci_name",
                "category",
                "yrcompiled",
                "google_link",
                "wiki_link",
                "iucn_link",
               # "legend",
                "dist_comm"
            ],
            aliases=[
                "Common name:",
                "Scientific name:",
                "Endangerment category:",
                "Year compiled:",
                "Google:",
                "Wikipedia:",
                "IUCN:",
                #"Regional catagory: ",
                "Comment: "
            ],
            localize=False
        ),

        tooltip=folium.GeoJsonTooltip(
            fields=["display_name", "sci_name"],# "legend"],
            aliases=["Name:", "Scientific name:"],# "Regional catagory: "],
            localize=True,
            sticky=True
        ),

        highlight_function=lambda feature: {
            "fillColor": "#ffff00",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.8,
        },

        popup_keep_highlighted=True,
    ).add_to(group)


ex_group = folium.FeatureGroup(name="Extinct", show=False).add_to(m)
cr_group = folium.FeatureGroup(name="Critically endangered", show=True).add_to(m)
en_group = folium.FeatureGroup(name="Endangered", show=False).add_to(m)
#vu_group = folium.FeatureGroup(name="Vunrable", show=False).add_to(m)


make_layer(ex, ex_group, fill_color="#626262")
make_layer(cr, cr_group, fill_color="#FF0000")
make_layer(en, en_group, fill_color="#0033FF")
#make_layer(vu, vu_group, fill_color="#C8FF00")




def stacked_opacity(n, base=0.1):
    return 1 - (1 - base) ** n

levels = [1, 2, 3, 4, 5, 6]
opacities = {n: stacked_opacity(n) for n in levels}


### Legend ###
legend_rows = ""

for n in reversed(levels):  # 6 at top
    label = "6" if n == 6 else str(n)
    op = opacities[n]

    legend_rows += f"""
    <div style="text-align:right;">{label}</div>

    <div style="width:20px;height:20px;
        background: rgba(0,51,255,{op});
        border:1px solid rgba(0,0,0,0.2);
        border-radius:3px;"></div>

    <div style="width:20px;height:20px;
        background: rgba(255,0,0,{op});
        border:1px solid rgba(0,0,0,0.2);
        border-radius:3px;"></div>
    """

legend_html = f"""
{{% macro html(this, kwargs) %}}

<div style="
    position: fixed;
    bottom: 40px;
    left: 40px;
    background: rgba(255, 255, 255, 0.7);
    padding: 12px;
    border-radius: 10px;
    border: 1px solid rgba(0,0,0,0.2);
    box-shadow: 0 2px 8px rgba(0,0,0,0.15);
    font-family: Arial;
    font-size: 12px;
    z-index: 9999;

    transform: scale(1.5);
    transform-origin: bottom left;
">

<div style="
    display: grid;
    grid-template-columns: 10px 15px 40px 28px;
    grid-row-gap: 5px;
    grid-column-gap: 6px;
    align-items: center;
">

    <!-- Vertical label -->
    <div style="
        grid-row: span 8;
        writing-mode: vertical-rl;
        transform: rotate(180deg);
        text-align: center;
        font-weight: bold;
        font-size: 11px;
    ">
        Number of species
    </div>

    <!-- Header -->
    <div></div>
    <div style="
        grid-column: span 2;
        text-align: center;
        font-weight: bold;
        font-size: 11px;
    ">
        Endangerment category
    </div>

    <!-- Column labels -->
    <div></div>
    <div style="display:flex; justify-content:left;"><b>EN</b></div>
    <div style="display:flex; justify-content:left;"><b>CR</b></div>

    {legend_rows}

</div>
</div>

{{% endmacro %}}
"""

legend = MacroElement()
legend._template = Template(legend_html)

m.get_root().add_child(legend)






startup_message = """
{% macro html(this, kwargs) %}

<div id="startup-box" style="
    position: fixed;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    width: 50vw;
    background: rgba(255,255,255,0.95);
    border: 1px solid #ccc;
    border-radius: 10px;
    padding: 12px;
    font-family: Arial;
    font-size: 20px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2);
    z-index: 9999;
">

    <div style="font-weight: bold; margin-bottom: 6px;">
        Welcome to heatmap of mamal species endangerment.
    </div>

    <img src="https://upload.wikimedia.org/wikipedia/commons/0/0b/Status_iucn2.3.svg"
     style="width:50%; border-radius:8px; margin-bottom:10px;">

    <div style="margin-bottom: 10px;">
        <div>This is a map showing endangered animals according to the IUCN red list. IUCN is a global organization that consolidates red-lists from different countries into one category. It categorizes the species into categories ranging from "Least concern" to "Extinct (show image of categories).</div></div>

        <div>This map shows layered habitats of non-marine endangered mammals in the Critically Endangered (CR) and Endangered (EN) categories. This produces a heatmap of the regions of endangerment. The different layers can be controlled in the top left corner, and this info screen can be found in the bottom left under "info"

        <div>For more information on IUCN and how to support animal concervation <a href="https://www.iucnredlist.org/about/background-history">Click here</a> or to see the source code <a href="https://github.com/igloo861/Heatmap-of-mammal-endangerment">Click here</a></div>


        
    </div>

    <button onclick="document.getElementById('startup-box').style.display='none';"
        style="
            padding: 4px 8px;
            font-size: 12px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
        ">
        Hide
    </button>

</div>

<button onclick="
    var box = document.getElementById('startup-box');
    box.style.display = (box.style.display === 'none') ? 'block' : 'none';
"
style="
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 9999;
">
Info
</button>

{% endmacro %}
"""


startup = MacroElement()
startup._template = Template(startup_message)

m.get_root().add_child(startup)





folium.LayerControl().add_to(m)

m.save("tmpmap_combined_simplified.html")
webbrowser.open("tmpmap_combined_simplified.html")


True

In [ ]:
tmp = en[en["sci_name"] == "Pteropus vampyrus"]

tmp.to_csv("tmp_rhino.csv")

In [ ]:
tmp

import geopandas as gpd

# assuming your GeoDataFrame is called gdf
gdf = tmp.set_geometry("geometry")

# dissolve geometries by species name
gdf_combined = gdf.dissolve(
    by="sci_name",
    aggfunc="first"  # keeps first value for other columns
).reset_index()